# Бонус. Сравнение с большой моделью (Qwen3-0.6B-Base)
Считаем $\log P$ тех же двух предложений из отложенной части (связного и перемешанного), что и в `lab01.ipynb`, у модели `Qwen/Qwen3-0.6B-Base` и сравниваем разрыв с тем, что дала наша биграммная модель.

**Что используется из `lab01.ipynb`:**
- веса биграммной модели `weights/bigram_<корпус>.json` (счётчики, сохранённые на этапе «Счётчики и вероятности»);
- отложенная часть корпуса `data/processed/<корпус>.json`: из неё теми же правилами и тем же `SEED` заново выбираются связное предложение и его перемешанный вариант. Предложения совпадают с теми, что были в `lab01.ipynb`.

Запускать после `lab01.ipynb`, из той же папки.

In [10]:
#!python -m pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.6.0+cu118
Uninstalling torch-2.6.0+cu118:
  Successfully uninstalled torch-2.6.0+cu118
Found existing installation: torchvision 0.21.0
Uninstalling torchvision-0.21.0:
  Successfully uninstalled torchvision-0.21.0
Found existing installation: torchaudio 2.6.0+cu118
Uninstalling torchaudio-2.6.0+cu118:
  Successfully uninstalled torchaudio-2.6.0+cu118


## Параметры и загрузка Qwen
Параметры должны совпадать с `lab01.ipynb`. Модель и токенизатор загружаются так же, как в исходном демо.

In [2]:
import os, re, json, math, random
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

CORPUS = "tolstoy"                 # корпус, на котором обучалась модель в lab01 (PRIMARY)
ALPHA = 0.1                        # сглаживание, как в lab01
SENT_MIN, SENT_MAX = 8, 15         # длина предложения, как в lab01
SEED = 42                          # seed перемешивания, как в lab01
PROC_DIR, WEIGHTS_DIR = "data/processed", "weights"
SENT_END = {".", "!", "?"}

MODEL_ID = os.environ.get("TGO_MODEL_ID", "Qwen/Qwen3-0.6B-Base")

device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print("device:", device)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)   # transformers 4.x
model = model.to(device).eval()
print(f"модель: {MODEL_ID}, параметров: {sum(p.numel() for p in model.parameters()) / 1e6:.0f} млн, словарь: {len(tok)} токенов")

device: cuda


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

модель: Qwen/Qwen3-0.6B-Base, параметров: 596 млн, словарь: 151669 токенов


## Два предложения из `lab01.ipynb`
Связное предложение это первое подходящее по длине в отложенной части, перемешанное - те же слова в случайном порядке (`random.Random(SEED)`, знак конца остаётся на месте). Правила выбора скопированы из `lab01.ipynb`.

In [5]:
def split_sentences(tokens):                       # как в lab01
    sents, cur = [], []
    for t in tokens:
        cur.append(t)
        if t in SENT_END:
            sents.append(cur); cur = []
    return sents + ([cur] if cur else [])

def detok(tokens):                                 # как в lab01: токены -> читаемая строка
    s = re.sub(r" ([.!?])", r"\1", " ".join(tokens))
    return re.sub(r"(^|[.!?] )([а-яё])", lambda m: m.group(1) + m.group(2).upper(), s)

test = json.load(open(f"{PROC_DIR}/{CORPUS}.json", encoding="utf-8"))["test"]   # отложенная часть
sent = next(s for s in split_sentences(test) if s[-1] in SENT_END and SENT_MIN <= len(s) - 1 <= SENT_MAX)
words = sent[:-1]
random.Random(SEED).shuffle(words)
coh, shuf = sent, words + sent[-1:]               # связное и перемешанное (списки токенов)

print("связное   :", detok(coh))
print("вперемешку:", detok(shuf))

связное   : Сказал степан аркадьич довольный тем что будут читать и дадут ему немножко опомниться.
вперемешку: Читать будут аркадьич дадут опомниться что немножко и довольный тем сказал степан ему.


## logP у биграммной модели (по сохранённым весам)
Из `weights/bigram_<корпус>.json` берутся счётчики `c(w)` и `c(v,w)`. Вероятность со сглаживанием Лапласа:

$$ P(w\mid v)=\frac{c(v,w)+\alpha}{c(v)+\alpha V}, \qquad \log P=\sum_t \log P(w_t\mid w_{t-1}) $$

Первое слово не оценивается, так как для него нет предыдущего слова, поэтому слагаемых на одно меньше, чем токенов.

In [8]:
weights = json.load(open(f"{WEIGHTS_DIR}/bigram_{CORPUS}.json", encoding="utf-8"))
bigram = weights["bigram"]                         # c(v,w)
V = len(weights["unigram"])                        # размер словаря обучающей части
ctx = {v: sum(c.values()) for v, c in bigram.items()}   # c(v) = сумма c(v,w) по w

def our_logprob(tokens, alpha):
    total = 0.0
    for v, w in zip(tokens, tokens[1:]):
        p = (bigram.get(v, {}).get(w, 0) + alpha) / (ctx.get(v, 0) + alpha * V)   # сглаженная P(w|v)
        total += math.log(p)
    return total

our_coh, our_shuf = our_logprob(coh, ALPHA), our_logprob(shuf, ALPHA)
n_pairs = len(coh) - 1                             # число оцениваемых слов
print(f"Реализованная модель (alpha = {ALPHA}): logP(связное) = {our_coh:.2f}, logP(вперемешку) = {our_shuf:.2f}")

Реализованная модель (alpha = 0.1): logP(связное) = -104.02, logP(вперемешку) = -119.48


## logP у Qwen3-0.6B-Base

In [11]:
#import torch
#print(torch.__version__, torch.version.cuda)
#print(torch.cuda.get_arch_list())

2.11.0+cu128 12.8
['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


In [13]:
@torch.no_grad()
def sequence_log_likelihood(text: str):
    ids = tok(text, return_tensors="pt").to(device)
    logits = model(**ids).logits[0, :-1].float()          # предсказания для токенов 1..T
    logp = torch.log_softmax(logits, dim=-1)              # логарифмы вероятностей по словарю
    target = ids["input_ids"][0, 1:]                      # настоящие следующие токены
    token_logp = logp[torch.arange(target.numel(), device=device), target].cpu()   # logP
    return token_logp.sum().item(), token_logp.numel()

q_coh, n_coh = sequence_log_likelihood(detok(coh))
q_shuf, n_shuf = sequence_log_likelihood(detok(shuf))
for name, t in (("связное", detok(coh)), ("вперемешку", detok(shuf))):
    ll, n = sequence_log_likelihood(t)
    print(f"{name:<12} log P = {ll:8.1f}   на токен = {ll / n:6.2f}   перплексия = {math.exp(-ll / n):8.1f}   токенов оценено: {n}")

связное      log P =   -135.6   на токен =  -4.52   перплексия =     91.9   токенов оценено: 30
вперемешку   log P =   -157.9   на токен =  -5.44   перплексия =    231.2   токенов оценено: 29


## Сравнение разрыва
Разрыв - это разница между $\log P$(связное) и $\log P$(перемешанное). Чем он больше, тем сильнее модель отличает осмысленный порядок слов от случайного.

In [16]:
print(f"{'модель':<24}{'logP связное':>14}{'logP вперемешку':>17}{'разрыв':>9}{'разрыв / токен':>16}")
print(f"{'bigram (alpha=' + str(ALPHA) + ')':<24}{our_coh:>14.2f}{our_shuf:>17.2f}{our_coh - our_shuf:>9.2f}{(our_coh - our_shuf) / n_pairs:>16.3f}")
print(f"{'Qwen3-0.6B-Base':<24}{q_coh:>14.2f}{q_shuf:>17.2f}{q_coh - q_shuf:>9.2f}{(q_coh - q_shuf) / (n_coh):>16.3f}")
print(f"\nоценено слов нашей моделью: {n_pairs}; токенов Qwen: {n_coh} (связное), {n_shuf} (вперемешку)")

модель                    logP связное  logP вперемешку   разрыв  разрыв / токен
bigram (alpha=0.1)             -104.02          -119.48    15.46           1.190
Qwen3-0.6B-Base                -135.62          -157.86    22.24           0.741

оценено слов нашей моделью: 13; токенов Qwen: 30 (связное), 29 (вперемешку)
